In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
len(documents)  

72

In [3]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.
Return a valid JSON object with this shape:
{"questions": ["question 1", "question 2", "question 3", "question 4", "question 5"]}

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [4]:
from dotenv import load_dotenv
import os
load_dotenv()
from groq import Groq
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [5]:
from evaluation_utils import llm_structured, llm_structured_retry

In [6]:
import json

In [7]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        client,
        data_gen_instructions,
        user_prompt
    )

    results = []
    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

In [8]:
doc=documents[0]
doc

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [9]:
generate_ground_truth(doc)

([{'question': 'What motivated the author to build everything from scratch in plain Python for this Retrieval-Augmented Generation (RAG) system?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'How do Large Language Models (LLMs) differ from the language models used in our phones?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What are some limitations of using LLMs in practice, according to the text?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'How does the RAG system help to mitigate the knowledge cutoff issue with LLMs?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What will the two main parts of this module on RAG cover?',
   'document': '01-agentic-rag/lessons/01-intro.md'}],
 CompletionUsage(completion_tokens=112, prompt_tokens=1069, total_tokens=1181, completion_time=0.205323453, completion_tokens_details=None, prompt_time=0.065388145, prompt_tokens_details=None, queue_time=0.018

In [10]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [11]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents[:3], generate_ground_truth)

#hit rate limit yikes

  0%|          | 0/3 [00:00<?, ?it/s]

In [12]:
questions_lists, usages = zip(*results)  # unzip the (results, usage) tuples

for doc, usage in zip(documents[:3], usages):
    print(doc["filename"], usage.prompt_tokens, usage.completion_tokens)

total_input = sum(u.prompt_tokens for u in usages)
total_output = sum(u.completion_tokens for u in usages)
print("Total input tokens:", total_input)
print("Total output tokens:", total_output)

01-agentic-rag/lessons/01-intro.md 1069 131
01-agentic-rag/lessons/02-environment.md 1342 108
01-agentic-rag/lessons/03-rag.md 1815 85
Total input tokens: 4226
Total output tokens: 324


In [13]:
import pandas as pd

In [14]:
df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [15]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [16]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
text_index.fit(chunks)

def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

In [17]:
from minsearch import VectorSearch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm  # plain tqdm, not tqdm.notebook — avoids ipywidgets/zmq sockets
import numpy as np

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

batch_size = 64
contents = [c["content"] for c in chunks]

chunk_vectors = []
for i in tqdm(range(0, len(contents), batch_size)):
    batch = contents[i:i + batch_size]
    batch_vectors = embedding_model.encode(batch, show_progress_bar=False)
    chunk_vectors.append(batch_vectors)

chunk_vectors = np.vstack(chunk_vectors)

vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(chunk_vectors, chunks)

def vector_search(query, num_results=5):
    query_vector = embedding_model.encode(query)
    return vector_index.search(query_vector, num_results=num_results)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

100%|██████████| 5/5 [00:18<00:00,  3.78s/it]


In [18]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [19]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [21]:
q = ground_truth[0]["question"]
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [27]:
text_results = text_search(q, num_results=5)
text_results

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [28]:
vector_results = vector_search(q, num_results=5)
vector_results

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [30]:
def compute_relevance(q, search_function):
    filename = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename))

    return relevance

In [31]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [32]:
compute_relevance(ground_truth[0], text_search)

[0, 0, 0, 0, 1]

In [33]:
relevance_total = compute_relevance_total(ground_truth, text_search)

100%|██████████| 360/360 [00:01<00:00, 316.93it/s]


In [34]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [ ]:
hit_rate(relevance_total)

0.7583333333333333

In [36]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [37]:
relevance_vector_total = compute_relevance_total(ground_truth, vector_search)

100%|██████████| 360/360 [00:09<00:00, 38.71it/s]


In [38]:
mrr(relevance_total)

0.5942592592592594

In [39]:
from functools import partial

k_values = [1, 50, 100, 200]
mrr_by_k = {}

for k in k_values:
    search_fn = partial(hybrid_search, k=k)
    relevance = compute_relevance_total(ground_truth, search_fn)
    mrr_by_k[k] = mrr(relevance)

best_k = max(mrr_by_k, key=mrr_by_k.get)
best_mrr = mrr_by_k[best_k]

print(mrr_by_k)
print(f"best k: {best_k}, best MRR: {best_mrr:.4f}")

best_k, best_mrr

100%|██████████| 360/360 [00:09<00:00, 36.02it/s]

{1: 0.6722685185185188, 50: 0.6721296296296295, 100: 0.6721296296296295, 200: 0.6721296296296295}
best k: 1, best MRR: 0.6723


(1, 0.6722685185185188)